# OPTIONAL - disabled by default
This notebook publishes `SUPPLIER_LINKS_PREDICTIONS` to SAP Business Data Cloud. It needs a
configured SAP BDC zero-copy connector (`CONNECTORS.ZEROCOPY.DEFAULT_SAP_BDC_CONNECTOR`),
which a new training account does not have. It is **not** part of the standard environment.

# Publish Data to SAP BDC via Zerocopy Connector
This notebook sets up an Iceberg V3 table and publishes it as a data product to SAP Business Data Cloud.

In [ ]:
%%sql -r connectors
SHOW ZEROCOPY CONNECTORS IN ACCOUNT;

In [ ]:
%%sql -r create_share
CREATE SHARE IF NOT EXISTS SNOWFLAKE_SUPPLIER_LINKS_PREDICTIONS_SHARE;

In [ ]:
%%sql -r grant_privs
GRANT USAGE ON DATABASE DEMO_19865_DB TO SHARE SNOWFLAKE_SUPPLIER_LINKS_PREDICTIONS_SHARE;
GRANT USAGE ON SCHEMA DEMO_19865_DB.GNN_SUPPLY_CHAIN_RISK TO SHARE SNOWFLAKE_SUPPLIER_LINKS_PREDICTIONS_SHARE;
GRANT SELECT ON TABLE DEMO_19865_DB.GNN_SUPPLY_CHAIN_RISK.SUPPLIER_LINKS_PREDICTIONS TO SHARE SNOWFLAKE_SUPPLIER_LINKS_PREDICTIONS_SHARE;

In [ ]:
%%sql -r add_share
ALTER ZEROCOPY CONNECTOR CONNECTORS.ZEROCOPY.DEFAULT_SAP_BDC_CONNECTOR ADD SHARE SNOWFLAKE_SUPPLIER_LINKS_PREDICTIONS_SHARE;

In [ ]:
%%sql -r connector_desc
DESC ZEROCOPY CONNECTOR CONNECTORS.ZEROCOPY.DEFAULT_SAP_BDC_CONNECTOR;

In [ ]:
%%sql -r publish_result
SELECT SYSTEM$SAP_PUBLISH_DATA_PRODUCT(
  'CONNECTORS.ZEROCOPY.DEFAULT_SAP_BDC_CONNECTOR',
  'SNOWFLAKE_SUPPLIER_LINKS_PREDICTIONS_SHARE',
  '{
    "title": "Snowflake Supplier Links Predictions",
    "shortDescription": "GNN-predicted hidden supplier relationships from Snowflake.",
    "description": "Contains graph neural network predictions of hidden supplier-to-supplier and supplier-to-material links in the supply chain, including probability scores and evidence strength."
  }',
  '{
    "csnInteropEffective": "1.2",
    "$version": "2.0",
    "meta": {
      "document": {
        "version": "1.0.0",
        "doc": "CSN document for GNN Supply Chain Link Predictions shared from Snowflake to SAP BDC."
      }
    },
    "definitions": {
      "GNN_SUPPLY_CHAIN_RISK": {
        "kind": "service",
        "doc": "Service exposing GNN-predicted supplier link entities."
      },
      "GNN_SUPPLY_CHAIN_RISK.SUPPLIER_LINKS_PREDICTIONS": {
        "kind": "entity",
        "doc": "GNN-predicted hidden relationships between supply chain nodes, including confidence scores and evidence metadata.",
        "@EndUserText.label": "Supplier Links Predictions",
        "elements": {
          "LinkID": {
            "doc": "Unique identifier for the predicted link.",
            "key": true,
            "type": "cds.Decimal",
            "precision": 10,
            "scale": 0
          },
          "SourceNodeID": {
            "doc": "Identifier of the source node in the supply chain graph.",
            "type": "cds.String"
          },
          "SourceNodeType": {
            "doc": "Type of the source node (e.g., Supplier, Material, Region).",
            "type": "cds.String"
          },
          "TargetNodeID": {
            "doc": "Identifier of the target node in the supply chain graph.",
            "type": "cds.String"
          },
          "TargetNodeType": {
            "doc": "Type of the target node (e.g., Supplier, Material, Region).",
            "type": "cds.String"
          },
          "LinkType": {
            "doc": "Classification of the predicted relationship (e.g., supplies_to, depends_on).",
            "type": "cds.String"
          },
          "Probability": {
            "doc": "GNN model confidence score for the predicted link (0 to 1).",
            "type": "cds.Decimal",
            "precision": 10,
            "scale": 0
          },
          "EvidenceStrength": {
            "doc": "Qualitative strength of supporting evidence (e.g., Strong, Moderate, Weak).",
            "type": "cds.String"
          },
          "SupportingData": {
            "doc": "Additional data or references supporting the prediction.",
            "type": "cds.String"
          },
          "ModelVersion": {
            "doc": "Version identifier of the GNN model that produced this prediction.",
            "type": "cds.String"
          },
          "PredictedAt": {
            "doc": "Timestamp when the prediction was generated.",
            "type": "cds.String"
          }
        }
      }
    }
  }'
);

In [ ]:
%%sql -r dataframe_1
